In [29]:
import pandas as pd
from datasets import Dataset
import os
import json
from google import genai
from google.genai import types

with open("dataset_postprocessed/train.nl.filtered", "r") as f:
    nl_lines = f.read().splitlines()
        
with open("dataset_postprocessed/train.cm.filtered", "r") as f:
    bash_lines = f.read().splitlines()

train_dataset = Dataset.from_dict({
    "nl": nl_lines,
    "bash": bash_lines
})

df = pd.DataFrame(train_dataset)
df["root_cmd"] = df["bash"].apply(lambda x: x.split()[0] if isinstance(x, str) and len(x.split()) > 0 else "unknown")

root_cmds = set(df["root_cmd"].unique())

#generated_roots = set([i.removeprefix("generated_").split(".")[0] for i in os.listdir("generated_jsons")])
generated_roots = set()
generated_roots.add("find")

In [30]:
# df[df["root_cmd"].isin(targets)]["root_cmd"].value_counts()
exclusion = set(["$(", "myprogram|more", "xargs", "|"])
targets = root_cmds ^ generated_roots ^ exclusion
print(f"Total root commands to process: {len(targets)}")
print(targets)

Total root commands to process: 105
{'exit', 'rm', 'finger', 'mv', 'apropos', 'mktemp', 'basename', 'read', 'diff', 'rev', 'pstree', 'which', 'cd', 'tar', 'shift', 'groups', 'w', 'sleep', 'hostname', 'history', 'grep', 'df', 'cut', 'head', 'ln', 'clear', 'man', 'gunzip', 'gzip', 'pwd', 'bzip2', 'mount', 'printf', 'date', 'wget', 'bunzip2', 'column', 'compress', 'bg', 'env', 'rsync', 'cal', 'watch', 'comm', 'sort', 'cp', 'join', 'tree', 'ifconfig', 'su', 'ssh-keygen', 'bash', 'fg', 'chown', 'ls', 'zcat', 'readlink', 'scp', 'unknown', 'tac', 'du', 'kill', 'ssh', 'touch', 'who', 'cat', 'top', 'wc', 'fold', 'curl', 'info', 'zless', 'mkdir', 'shopt', 'nl', 'md5', 'ps', 'paste', 'jobs', 'whoami', 'ping', 'dirname', 'chgrp', 'crontab', 'seq', 'rmdir', 'uniq', 'tee', 'od', 'yes', 'split', 'uname', 'rename', 'logout', 'shred', 'stat', 'tr', 'tail', 'cpio', 'file', 'echo', 'chmod', 'md5sum', 'bind', 'dig'}


In [31]:
# https://ai.google.dev/gemini-api/docs/batch-api?batch=file

In [32]:
batch_filename = "batch_requests_.jsonl"
from google.genai import types
from pydantic import BaseModel

# https://ai.google.dev/gemini-api/docs/structured-output?example=recipe
class Command(BaseModel):
    bash: str
    nl: str

with open(batch_filename, "w") as f:
    for root in list(targets):
        prompt = f"""Generate 400 unique bash commands starting with the root command '{root}'. 
        For each command, provide a brief corresponding natural language description. Use varied language for the descriptions.
        Output the result strictly as a valid JSON array of objects, with each object containing 'bash' and 'nl' keys.
        Example:
        [
            {{"bash": "{root} --help", "nl": "Show help for the {root} command"}}
        ]
        """
        
        request_line = {
            "key": root, 
            "request": {
                "contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {
                    "responseMimeType": "application/json",
                    "responseSchema": {
        "type": "ARRAY",
        "description": "A list of bash commands and their natural language descriptions.",
        "items": {
            "type": "OBJECT",
            "properties": {
                "bash": {
                    "type": "STRING",
                    "description": "The perfectly escaped bash command."
                },
                "nl": {
                    "type": "STRING",
                    "description": "A brief explanation of what the command does."
                }
            },
            "required": ["bash", "nl"] 
        }
    }
                }
            }
        }

        f.write(json.dumps(request_line) + "\n")

print(f"Created {batch_filename} with {len(targets)} requests.")


Created batch_requests_.jsonl with 105 requests.


In [ ]:
client = genai.Client()

uploaded_file = client.files.upload(file = batch_filename, 
                                    config = types.UploadFileConfig(display_name = 'my-batch-requests-', mime_type='jsonl'))


In [ ]:
batch_job = client.batches.create(
    model = "gemini-3.1-flash-lite-preview",
    src = uploaded_file.name
)

print(f"job name {batch_job.name}")
print(f"status: {batch_job.state}")

In [39]:
import json
import os
# from google import genai

client = genai.Client()

JOB_NAME = batch_job.name 

batch_job = client.batches.get(name=JOB_NAME)
print(f"Current Status: {batch_job.state}")


Current Status: JobState.JOB_STATE_SUCCEEDED


In [ ]:
batch_jobs = client.batches.list()

# Optional query config:
# batch_jobs = client.batches.list(config={'page_size': 5})

for b in batch_jobs:
    print(b)
    pass


In [ ]:
# copied https://ai.google.dev/gemini-api/docs/batch-api?batch=file#batch-job-status
if batch_job.state.name == 'JOB_STATE_SUCCEEDED':

    # If batch job was created with a file
    if batch_job.dest and batch_job.dest.file_name:
        # Results are in a file
        result_file_name = batch_job.dest.file_name
        print(f"Results are in file: {result_file_name}")

        print("Downloading result file content...")
        file_content = client.files.download(file=result_file_name)
        # Process file_content (bytes) as needed
        print(file_content.decode('utf-8'))

    # If batch job was created with inline request
    # (for embeddings, use batch_job.dest.inlined_embed_content_responses)
    elif batch_job.dest and batch_job.dest.inlined_responses:
        # Results are inline
        print("Results are inline:")
        for i, inline_response in enumerate(batch_job.dest.inlined_responses):
            print(f"Response {i+1}:")
            if inline_response.response:
                # Accessing response, structure may vary.
                try:
                    print(inline_response.response.text)
                except AttributeError:
                    print(inline_response.response) # Fallback
            elif inline_response.error:
                print(f"Error: {inline_response.error}")
    else:
        print("No results found (neither file nor inline).")
else:
    print(f"Job did not succeed. Final state: {batch_job.state.name}")
    if batch_job.error:
        print(f"Error: {batch_job.error}")

In [ ]:
invalids = []

for line in file_content.decode('utf-8').splitlines():
    data = json.loads(line)
    root = data.get("key")
    
    try:
        response_text = data["response"]["candidates"][0]["content"]["parts"][0]["text"]
        commands_array = json.loads(response_text)

        with open(f"generated_jsons_1/generated_{root}.json", "w") as json_file:
            json.dump(commands_array, json_file, indent=2)
        
        nl_lines = []
        bash_lines = []
        
        for item in commands_array:
            bash_lines.append(item.get("bash", "").strip())
            nl_lines.append(item.get("nl", "").strip())
        
        if len(nl_lines) != len(bash_lines):
            print(f"{root} not same lengths?: {len(nl_lines)}, {len(bash_lines)}")
            invalids.append(root)

        # Write to respective files
        with open(f"generated_data_1/generated_{root}.nl", "w") as nl_file:
            nl_file.write("\n".join(nl_lines))
        
        with open(f"generated_data_1/generated_{root}.cm", "w") as cm_file:
            cm_file.write("\n".join([i.replace("\n", "\\n").replace("\r", "\\r") for i in bash_lines]))
            
        print(f"Successfully saved files for root: {root}")
        
    except Exception as e:
        print(f"Failed to parse output for root '{root}': {e}")

Successfully saved files for root: echo
Successfully saved files for root: chmod
Successfully saved files for root: md5sum
Successfully saved files for root: bind
Successfully saved files for root: dig
Successfully saved files for root: exit
Successfully saved files for root: rm
Successfully saved files for root: finger
Successfully saved files for root: mv
Successfully saved files for root: apropos
Successfully saved files for root: mktemp
Successfully saved files for root: basename
Successfully saved files for root: read
Successfully saved files for root: diff
Successfully saved files for root: rev
Successfully saved files for root: pstree
Successfully saved files for root: which
Successfully saved files for root: cd
Successfully saved files for root: tar
Successfully saved files for root: shift
Successfully saved files for root: groups
Successfully saved files for root: w
Successfully saved files for root: sleep
Successfully saved files for root: hostname
Successfully saved files fo